# 03 — Entrega y reanudación

Trabaja sobre lo que ya hay en Drive: verifica el último lote, empaqueta un ZIP para descargar y **reanuda los reels cortados** reutilizando el guion y las imágenes cacheadas.

Se puede ejecutar en una sesión nueva sin volver a generar nada.


In [ ]:
#@title 1. Localizar el último lote en Drive { display-mode: "form" }
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/canal-estoicismo")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import scripts.colab_env as env

candidates = sorted(
    (path for path in (env.DRIVE_ROOT / "output").glob("lote_*") if path.is_dir()),
    key=lambda path: path.name,
)
if not candidates:
    raise SystemExit("No hay lotes en Drive todavía: ejecuta primero 02_lote_reels.ipynb.")

RUN_DIR = candidates[-1]
print("lotes encontrados:", [path.name for path in candidates])
print("lote a procesar  :", RUN_DIR)


In [ ]:
#@title 2. Verificar los vídeos del lote { display-mode: "form" }
rows = env.verify_videos(RUN_DIR, "reel")
manifest = env.write_manifest(RUN_DIR, rows, extra={"verified": True})
print(f"\nmanifest: {manifest}")


In [ ]:
#@title 3. Empaquetar un ZIP { display-mode: "form" }
import zipfile

zip_path = RUN_DIR / (RUN_DIR.name + ".zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for item in sorted(RUN_DIR.iterdir()):
        if item.suffix in (".mp4", ".json") and item.is_file():
            archive.write(item, item.name)

print(f"ZIP: {zip_path.name}  ({zip_path.stat().st_size / 1e6:.1f} MB)")
for name in sorted(zipfile.ZipFile(zip_path).namelist()):
    print("  ", name)


In [ ]:
#@title 4. Descargar el ZIP { display-mode: "form" }
from google.colab import files

files.download(str(zip_path))
print("Descarga iniciada:", zip_path.name)
print("Si el navegador la bloquea, el ZIP también está en tu Drive.")


In [ ]:
#@title 5. Reanudar reels cortados (usa guion + caché) { display-mode: "form" }
#@markdown Ejecutar la reanudación ahora:
RESUME = False #@param {type:"boolean"}

orphans = []
for story in sorted(RUN_DIR.glob("*.story.json")):
    video = RUN_DIR / story.name.replace(".story.json", ".mp4")
    if not video.exists():
        orphans.append((story, video))

if not orphans:
    print("No hay reels cortados en este lote.")
else:
    print(f"{len(orphans)} reel(s) sin terminar:\n")
    for story, video in orphans:
        print(f"  guion: {story.name}")
        print(f"  salida: {video.name}")

    if RESUME:
        for story, video in orphans:
            print(f"\n--- reanudando {video.name} ---")
            code = env.run(
                [
                    sys.executable, "main.py", "generate",
                    "--story-json", str(story),
                    "--output", str(video),
                    "--format", "reel",
                    "--profile", "auto",
                    "--image-provider", "sdxl_local",
                    "--cinematic",
                    "--visual-engine",
                ],
                cwd=env.PROJECT_DIR,
            )
            print(f"  exit={code} -> {'OK' if code == 0 else 'revisar'}")
        rows = env.verify_videos(RUN_DIR, "reel")
        env.write_manifest(RUN_DIR, rows, extra={"resumed": True})
    else:
        print("\nMarca RESUME=True y vuelve a ejecutar esta celda para completarlos.")
        print("Reutiliza el guion (--story-json) y las imágenes ya cacheadas: es lo más barato.")


## Hecho

- Los reels están en `Mi unidad/canal-estoicismo/output/<lote>/`.
- Cada vídeo tiene su `.story.json`: reutilízalo para rehacerlo sin gastar LLM ni generar imágenes nuevas.
- `manifest.json` guarda la verificación de este lote (resolución, duración, audio).

### Antes de publicar

1. Mira cada reel completo (sobre todo el audio y los subtítulos).
2. Comprueba que la duración encaja con el formato que publicas.
3. Recuerda: el `.story.json` es el activo reutilizable. Guárdalo siempre.
